# Sección 1 · Del VCF a una matriz limpia

*Control de calidad de datos SNP: leer, diagnosticar y filtrar — genética de poblaciones en* Nothofagus pumilio *(lenga)*

> **Cómo usar este notebook:** ejecuta las celdas en orden (Shift+Enter). Las celdas de ejercicio están casi vacías para que las completes. Al final está la **Tarea**, que revisaremos al comienzo de la próxima sección.
>
> **Esta sección en una frase:** aquí todavía no hacemos genética de poblaciones, hacemos **control de calidad de datos**. Hay dos conceptos (MAF y LD) que **no** se trabajan aquí: se justifican en las Secciones 2 y 3.


In [25]:
# ---- Preparación (ejecutar primero) ----
# Tamaño de gráficos en Jupyter
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 110)

source("R/helpers.R")
if (!file.exists("data/lenga_sim.vcf")) source("00_simular_datos.R")

## Objetivos de aprendizaje

Al terminar esta sección sabrás:

1. **Leer** un archivo VCF y reconocer sus tres partes: metadatos, cabecera de columnas y filas de variantes.
2. **Convertir** los genotipos en una matriz de dosis alélicas (0/1/2/NA) y **verificar** que esté alineada con la tabla de muestras.
3. **Diagnosticar** los datos faltantes por individuo y por SNP, y **aplicar** un umbral justificado.
4. **Guardar** la matriz limpia que usarán las secciones siguientes.

> **Dos conceptos quedan para después, a propósito:** el umbral de **MAF** se justifica en la Sección 2 (cuando medimos su efecto sobre la diversidad) y el de **LD** en la Sección 3 (cuando se podan SNP redundantes antes de analizar estructura).


## Conceptos previos

- **SNP** (*Single Nucleotide Polymorphism*) — posición del genoma en la que individuos distintos llevan nucleótidos diferentes. Es el marcador estándar de la genómica de poblaciones.
- **Genotipo vs. dosis alélica.** El genotipo describe qué alelos lleva un individuo en un locus (p. ej. `A/A`, `A/G`, `G/G`). La **dosis** es un número que cuenta cuántas copias del alelo alternativo (ALT) lleva: `0`, `1` o `2`. Es la representación con la que trabajaremos en casi todo el curso.
- **VCF** (*Variant Call Format*) — formato estándar para almacenar variantes. En un proyecto real el VCF sale de un *pipeline* (mapeo → llamado de variantes → filtros). Ese pipeline ya aplicó un primer filtro, **pero no es el filtro de tu pregunta biológica**.
- **Control de calidad (QC)** — diagnosticar y filtrar datos para que tus conclusiones no sean artefactos.

**Pregunta para anticipar:** si un individuo tiene el 40 % de sus genotipos faltantes, ¿qué crees que le pasa a su heterocigosidad? *(La respondemos al final de la sección.)*


## 1. Qué hay dentro de un VCF

Cada fila es un SNP; cada columna (desde `FORMAT`) es un individuo. El campo `GT` es el genotipo de cada individuo:

- `0/0` → homocigoto del alelo de referencia (REF)
- `0/1` → heterocigoto (una copia REF y una ALT)
- `1/1` → homocigoto del alelo alternativo (ALT)
- `./.` → genotipo **faltante** (no se pudo llamar)

Miremos las primeras líneas del archivo:


In [26]:
lineas <- readLines("data/lenga_sim.vcf", n = 30)
es_meta   <- grepl("^##", lineas)      # metadatos
es_header <- grepl("^#CHROM", lineas)  # cabecera de columnas
datos     <- lineas[!es_meta & !es_header]

cat("=== Metadatos (líneas ##) ===\n")
cat(substr(lineas[es_meta][1:3], 1, 80), sep = "\n")
cat("...\n\n")

cat("=== Cabecera (#CHROM + nombre de cada individuo) ===\n")
cat(substr(lineas[es_header], 1, 160), "\n...\n\n")

cat("=== Filas de variantes (una por SNP) ===\n")
for (d in datos[1:2]) cat(substr(d, 1, 130), "...\n")


=== Metadatos (líneas ##) ===
##fileformat=VCFv4.2
##source=simulacion_lenga_curso
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
...

=== Cabecera (#CHROM + nombre de cada individuo) ===
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	L01_01	L01_02	L01_03	L01_04	L01_05	L01_06	L01_07	L01_08	L01_09	L01_10	L01_11	L01_12	L02_01	L02_02	L02_03	L02_04	L0 
...

=== Filas de variantes (una por SNP) ===
chr1	520422	snp0001	C	T	.	PASS	.	GT	0/0	0/1	0/0	0/0	0/0	0/0	0/0	0/0	0/0	0/0	0/0	0/0	0/0	0/0	1/1	0/1	0/1	0/1	0/0	0/0	0/1	0/1	0/1	0/ ...
chr1	821710	snp0002	T	C	.	PASS	.	GT	0/1	0/1	0/1	0/1	1/1	1/1	1/1	0/1	0/1	0/0	0/1	1/1	0/1	0/0	1/1	0/1	1/1	0/1	0/0	1/1	1/1	0/1	0/1	0/ ...


Las líneas que empiezan con `##` son **metadatos** (formato, contigs). La línea `#CHROM` es la cabecera de columnas. A partir de ahí, cada fila es un SNP con su posición (`POS`), sus alelos (`REF`/`ALT`) y el genotipo de cada individuo.

**Para ver un VCF real:** el repo trae `datos_ejemplo/lenga_ejemplo.vcf`, un VCF real de lenga (Sekely et al. 2024) recortado a 3 sitios × 6 individuos × 301 SNP. La explicación de **cada columna** (`CHROM`, `POS`, `REF`, `ALT`, `QUAL`, `FILTER`, `INFO`, `FORMAT`, `GT`…) está en [`docs/formato_vcf.md`](docs/formato_vcf.md).


## 2. Importar y convertir a dosis: la matriz de dosis alélicas

**¿Qué es la dosis alélica?** Es un número que cuenta cuántas copias del alelo alternativo (ALT) lleva un individuo en un SNP:

| Genotipo | Significado | Dosis |
|---|---|---|
| `0/0` | homocigoto REF | `0` |
| `0/1` (o `1/0`) | heterocigoto | `1` |
| `1/1` | homocigoto ALT | `2` |
| `./.` | faltante | `NA` |

**¿Por qué dosis y no el genotipo en texto?** Porque la dosis es un **número**, y con números podemos hacer aritmética:

- La **frecuencia del alelo ALT** en un SNP es `mean(G[, j]) / 2` (la dosis promedio dividida entre 2).
- La **heterocigosidad** se calcula contando cuántas dosis valen `1`.
- Métodos como PCA, Fst o RDA trabajan directamente con la matriz de dosis.

**Nota sobre ploidía:** asumimos individuos **diploides**, por eso la dosis va de 0 a 2. Si un genotipo está faseado (`0|1` en vez de `0/1`) lo tratamos igual: contamos alelos, no haplotipos.

El primer paso es llevar el VCF a una matriz **individuos × SNP** (`G`), con una fila por individuo y una columna por SNP, y la dosis en cada celda. `read_vcf_dosage()` (en `R/helpers.R`) hace esa conversión con `vcfR`.

**Regla de oro:** cada fila de `G` debe corresponder a la *misma* fila de `meta`. El `stopifnot()` de abajo lo verifica; si alguna vez lo quitáramos, todo se calcularía «bien» y todo estaría mal, sin avisar.


In [ ]:
x <- read_vcf_dosage("data/lenga_sim.vcf")
G <- x$G
dim(G)            # individuos x SNP
G[1:5, 1:8]

meta <- read.csv("data/lenga_sim_muestras.csv", stringsAsFactors = FALSE)
meta <- meta[match(rownames(G), meta$id), ]
stopifnot(all(rownames(G) == meta$id))      # ¡siempre verificar el orden!
head(meta)

## 3. Dónde están las muestras (contexto, sin ejercicio)

Antes de analizar conviene **ver** el diseño: ¿dónde está cada población? Aquí hay 20 sitios a lo largo de ~2.000 km de latitud, con muestreo parejo (12 individuos por sitio).

> **Pregunta:** ¿qué sesgo introduciría un muestreo muy desigual entre sitios? *(Guárdala: reaparece en la Sección 2, con los alelos privados.)*

> **Solo contexto:** este mapa no es un paso de control de calidad. Sirve para ver el diseño (20 sitios, ~2.000 km). La pregunta de abajo reaparece en la Sección 2 con los alelos privados.


In [ ]:
pops <- aggregate(cbind(lat, lon, alt) ~ pop, data = meta, FUN = mean)
pops$n <- as.integer(table(meta$pop)[pops$pop])
ggplot(pops, aes(lon, lat, size = n, colour = alt)) +
  geom_point() + geom_text(aes(label = pop), nudge_x = 0.35, size = 3, colour = "grey30", show.legend = FALSE) +
  scale_colour_viridis_c() + labs(x = "Longitud", y = "Latitud", colour = "Altitud (m)", size = "n") +
  coord_fixed(1.2)

## 4. Diagnóstico de datos faltantes

El **call rate** es la proporción de genotipos llamados (lo opuesto al dato faltante). Se calcula por individuo y por SNP:

- Por **individuo**: muchos faltantes suelen ser señal de muestra de mala calidad (poco ADN, degradación).
- Por **SNP**: muchos faltantes suelen ser regiones difíciles de genotipar (repeticiones, baja cobertura).

**¿Qué esperas?** Mira los histogramas antes de decidir dónde poner el umbral. No hay un número «correcto»: hay un número *justificado*.

> **Si vienes de datos de expresión:** esto es exactamente el mismo ejercicio que filtrar células con pocos genes, o genes detectados en pocas células. Cambian los nombres, no la lógica.


In [ ]:
ind_cr <- rowMeans(!is.na(G))                 # call rate por individuo
snp    <- snp_summary(G)

ggplot(data.frame(ind_cr), aes(ind_cr)) + geom_histogram(bins = 30, fill = "steelblue") +
  geom_vline(xintercept = 0.8, lty = 2, colour = "red") +
  labs(x = "Proporción de genotipos llamados (individuo)", y = "N° de individuos")

ggplot(snp, aes(call_rate)) + geom_histogram(bins = 30, fill = "darkorange") +
  geom_vline(xintercept = 0.9, lty = 2, colour = "red") +
  labs(x = "Proporción de genotipos llamados (SNP)", y = "N° de SNP")

head(sort(ind_cr), 8)      # los peores individuos

## 5. Filtrado y guardado

`qc_filter()` aplica tres umbrales: **call rate por individuo**, **call rate por SNP** y **MAF mínima**. Hoy decidimos y justificamos los **dos primeros**, que son el objeto de esta sección (los datos faltantes). El tercero queda fijado en el valor del curso y lo justificaremos en la Sección 2, cuando veamos cuánto cambia la diversidad estimada según ese umbral.

**Idea clave:** los filtros son **decisiones científicas con consecuencias**, no un trámite. Por eso se escriben en Métodos. Fíjate en cuánto se pierde en cada etapa.


In [ ]:
q <- qc_filter(G, ind_cr = 0.80, snp_cr = 0.90, maf = 0.05)
G_clean    <- q$G
meta_clean <- meta[q$keep_ind, ]

resumen <- data.frame(
  etapa = c("Original", "Filtrado"),
  individuos = c(nrow(G), nrow(G_clean)),
  snps = c(ncol(G), ncol(G_clean)),
  faltante_pct = round(100 * c(mean(is.na(G)), mean(is.na(G_clean))), 2))
resumen

saveRDS(G_clean, "data/geno_clean.rds")
saveRDS(meta_clean, "data/meta_clean.rds")
table(meta_clean$pop)

## 6. Lo que queda pendiente (y dónde se cierra)

| Queda pendiente | Dónde se cierra |
|---|---|
| **Justificar el umbral de MAF** | Sección 2: se mide cómo cambia He según el umbral. La tabla de sensibilidad de la Tarea es el insumo. |
| **Ligamiento (LD)**: qué SNP son información redundante | Sección 3: se poda por LD antes de analizar **estructura** (PCA/DAPC, Fst). En diversidad, el LD es solo un diagnóstico. |
| **Exceso de heterocigotos** (posible paralogía) | Sección 2, como parte del diagnóstico de diversidad. |


---

## Ejercicios

**E1.** ¿Cuántos individuos y SNP se pierden con `ind_cr = 0.80`? Identifica los individuos descartados y verifica si pertenecen a una misma población.

> **Idea clave:** si la pérdida se concentra en una población, el filtro introduce sesgo de muestreo.


In [ ]:
# TU CÓDIGO AQUÍ

**E2.** ¿Importa el **orden** de los filtros (primero individuos, o primero SNP)? Reescribe la lógica del filtro (sin tocar `R/helpers.R`) y compara cuántos individuos y cuántos SNP quedan en cada caso.

> **Idea clave:** el orden cambia *qué* individuos se descartan. Si documentas el orden que usaste, el resultado es reproducible; si no, es irrepetible.

In [ ]:
# TU CÓDIGO AQUÍ

## Errores y dudas frecuentes

- **«Me sale `cannot open file 'R/helpers.R'`»** → Jupyter se abrió desde otra carpeta. Ciérralo y ejecútalo dentro de la carpeta del curso.
- **«El kernel no es R»** → cámbialo arriba a la derecha (debe decir **R**).
- **«¿Qué es ese `maf = 0.05` del filtro?»** → es un umbral sobre la frecuencia del alelo menos común. Queda fijo hoy y se justifica en la Sección 2.
- **«¿Y los SNP ligados?»** → no los tocamos hoy: se podan por LD en la Sección 3.
- **«¿Puedo eliminar individuos a mano?»** → puedes, pero debe quedar un criterio explícito. «Porque se veía raro» no es un método.


## Resumen: ideas que debes llevarte

1. Un VCF es una tabla de variantes; lo convertimos a una **matriz de dosis** (individuos × SNP, 0/1/2/NA).
2. `G` y `meta` deben estar **alineados por fila**; verifícalo siempre.
3. El QC de esta sección es de **datos faltantes**: call rate por individuo y por SNP.
4. Cada umbral es **una decisión que va en Métodos**. Los otros dos conceptos (MAF y LD) tienen su lugar en las Secciones 2 y 3.

> *Respuesta a la pregunta de anticipación:* un individuo con mucho dato faltante suele tener la heterocigosidad **sesgada a la baja** (los heterocigotos se llaman peor con poca cobertura), lo que distorsiona cualquier comparación entre poblaciones.


## Tarea (se revisa al comienzo de la próxima sección)

1. Escribe en **5 líneas** los **dos umbrales de datos faltantes** que usarás (por individuo y por SNP) y su justificación. Con eso ya tienes el inicio de tu sección *Métodos*.
2. **Tabla de sensibilidad:** cuenta cuántos individuos y cuántos SNP quedan para `ind_cr` en {0.7, 0.8, 0.9} × `snp_cr` en {0.8, 0.9, 0.95}. ¿Con qué combinación te quedas y por qué?

*Pregunta de reflexión:* tu proyecto trata de diversidad y estructura. ¿Qué sesgo puede introducir el filtro de faltantes sobre esas dos cantidades? (Pista: ¿a quién eliminas — al azar, o a una población entera?)
